# CKA Result Visualization

This notebook reads `.pt` result files produced by `analysis/compute_cka.py` and visualizes:

- within-model layer-by-layer CKA
- cross-model same-layer CKA (`standard` vs `attnres`)
- same-layer CKA across checkpoints for the same model
- optional cross-all CKA heatmaps

Each figure corresponds to a different analysis question:

- within-model heatmaps: how similar residual states remain across depth within one model
- cross-model same-layer curves: how similarly two architectures organize representations at matched depth
- across-checkpoint curves: how stable a model's same-layer geometry is across checkpoints
- cross-all heatmaps: which layers align best across two artifacts, even when depth does not match exactly

Interpretation should remain conservative:

- CKA measures representational similarity, not performance.
- Higher CKA means more similar geometry across aligned samples.
- Lower CKA means more divergent representation structure.
- These plots can be consistent with stronger or weaker layerwise differentiation, but they do not by themselves prove causality or superiority.


## 2. Imports

In [ ]:
from collections.abc import Mapping
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display

try:
    import pandas as pd
except ImportError:
    pd = None

plt.style.use("default")
plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
})


## 3. Config / Paths

In [ ]:
RESULTS_DIR = Path("artifacts/cka")
FIG_DIR = Path("artifacts/figures/cka")
RUN_TAG = ""


def with_run_tag(filename: str) -> str:
    """Apply an optional filename prefix for lookup/export convenience."""
    return f"{RUN_TAG}_{filename}" if RUN_TAG else filename


def result_file(filename: str) -> Path:
    """Build a default result path from the configured result directory."""
    return RESULTS_DIR / with_run_tag(filename)


# Default paths built from RESULTS_DIR / RUN_TAG.
# Override any individual variable below if you want to point at a custom file.
WITHIN_STANDARD = result_file("standard_last_within.pt")
WITHIN_ATTNRES = result_file("attnres_last_within.pt")
CROSS_SAME_LAYER = result_file("standard_vs_attnres_same_layer.pt")
STEP_COMPARE_STANDARD = result_file("standard_step1000_vs_step2000.pt")
STEP_COMPARE_ATTNRES = result_file("attnres_step1000_vs_step2000.pt")
CROSS_ALL_STANDARD_ATTNRES = result_file("standard_vs_attnres_cross_all.pt")

SAVE_DIR = FIG_DIR
SAVE_FIGURES = False
FIG_DPI = 180
SHOW_VALUES_IN_HEATMAP = False

HEATMAP_VMIN = 0.0
HEATMAP_VMAX = 1.0
HEATMAP_CMAP = "cividis"

CHECKPOINT_STANDARD_TITLE = "Same-layer CKA across checkpoints: standard"
CHECKPOINT_ATTNRES_TITLE = "Same-layer CKA across checkpoints: attnres"

RESULT_PATHS = {
    "within_standard": WITHIN_STANDARD,
    "within_attnres": WITHIN_ATTNRES,
    "cross_same_layer": CROSS_SAME_LAYER,
    "checkpoint_standard": STEP_COMPARE_STANDARD,
    "checkpoint_attnres": STEP_COMPARE_ATTNRES,
    "cross_all_standard_attnres": CROSS_ALL_STANDARD_ATTNRES,
}

SAVED_FIGURES = []
RESULTS = {}


## 4. Utility Functions

In [ ]:
def warn(message: str) -> None:
    """Print a notebook-friendly warning without stopping execution."""
    print(f"[warning] {message}")


def ensure_dir(path: Path | str) -> Path:
    """Create a directory if needed and return it as a Path."""
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def load_pt(path: Path | str):
    """Load a .pt file safely, returning None on failure."""
    path = Path(path)
    if not path.is_file():
        warn(f"Missing file: {path}")
        return None
    try:
        result = torch.load(path, map_location="cpu", weights_only=False)
    except Exception as exc:
        warn(f"Failed to load {path}: {exc}")
        return None
    if not isinstance(result, Mapping):
        warn(f"Expected a mapping in {path}, got {type(result).__name__}.")
        return None
    return result


def to_numpy(value):
    """Convert tensors or lists to a CPU numpy array."""
    if value is None:
        return None
    if isinstance(value, np.ndarray):
        return value
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def shape_or_none(value):
    """Return a safe tuple shape summary when possible."""
    if value is None:
        return None
    try:
        return tuple(to_numpy(value).shape)
    except Exception:
        return "unreadable"


def pretty_state_name(name: str) -> str:
    """Map canonical residual-state ids to paper-friendly display labels."""
    if name == "resid_final":
        return "Final residual"
    if isinstance(name, str) and name.startswith("resid_"):
        suffix = name.split("_", maxsplit=1)[1]
        if suffix.isdigit():
            return f"Block {int(suffix)} input"
    return str(name)


def format_state_list(states, max_items: int = 8) -> str:
    """Format state-name lists compactly for inspection tables."""
    states = [pretty_state_name(state) for state in list(states or [])]
    if not states:
        return "-"
    if len(states) <= max_items:
        return ", ".join(states)
    head = ", ".join(states[:max_items])
    return f"{head}, ... ({len(states)} total)"


def get_meta(result) -> dict:
    """Return the result metadata mapping, or an empty dict when absent."""
    if not isinstance(result, Mapping):
        return {}
    meta = result.get("meta", {})
    return dict(meta) if isinstance(meta, Mapping) else {}


def infer_mode(result) -> str:
    """Infer the result mode from metadata when possible."""
    return str(get_meta(result).get("mode", "unknown"))


def safe_title_from_meta(result) -> str:
    """Build a compact descriptive tag from result metadata for logs/tables."""
    meta = get_meta(result)
    parts = []
    mode = meta.get("mode")
    if mode:
        parts.append(f"mode={mode}")
    num_samples = meta.get("num_samples_used")
    if num_samples is not None:
        parts.append(f"n={num_samples}")
    group_filter = meta.get("group_filter")
    if group_filter:
        parts.append(f"group={group_filter}")
    return " | ".join(parts) if parts else "-"


def get_row_col_states(result):
    """Read state-name lists from the top level or fallback metadata."""
    meta = get_meta(result)
    row_states = list(result.get("row_states") or meta.get("states_a") or [])
    col_states = list(result.get("col_states") or meta.get("states_b") or [])
    return row_states, col_states


def validate_matrix_and_states(matrix, row_states, col_states, label: str = "result"):
    """Validate matrix shape against row/column state names."""
    matrix_np = to_numpy(matrix)
    if matrix_np is None:
        raise ValueError(f"{label}: missing matrix.")
    if matrix_np.ndim != 2:
        raise ValueError(f"{label}: matrix must be 2D, got shape {matrix_np.shape}.")
    if row_states and matrix_np.shape[0] != len(row_states):
        raise ValueError(
            f"{label}: matrix rows {matrix_np.shape[0]} != len(row_states) {len(row_states)}."
        )
    if col_states and matrix_np.shape[1] != len(col_states):
        raise ValueError(
            f"{label}: matrix cols {matrix_np.shape[1]} != len(col_states) {len(col_states)}."
        )
    return matrix_np.astype(float, copy=False)


def extract_within_matrix(result, label: str = "within"):
    """Extract a within-model heatmap payload from a compute_cka result."""
    if result is None:
        return None
    try:
        meta = get_meta(result)
        if meta.get("matrix_semantics") == "diag_only_same_layer_matrix":
            raise ValueError("diag-only same-layer matrix cannot be used as a within-model heatmap.")
        row_states, col_states = get_row_col_states(result)
        matrix = validate_matrix_and_states(result.get("matrix"), row_states, col_states, label=label)
        if not row_states and not col_states:
            row_states = [f"state_{idx}" for idx in range(matrix.shape[0])]
            col_states = [f"state_{idx}" for idx in range(matrix.shape[1])]
            warn(f"{label}: state names were missing; generated placeholder names.")
        elif row_states and not col_states:
            col_states = list(row_states)
        elif col_states and not row_states:
            row_states = list(col_states)
        return {
            "matrix": matrix,
            "row_states": row_states,
            "col_states": col_states,
            "meta": meta,
        }
    except Exception as exc:
        warn(f"{label}: could not parse within result: {exc}")
        return None


def extract_cross_all_matrix(result, label: str = "cross_all"):
    """Extract a full cross-artifact heatmap payload from a compute_cka result."""
    if result is None:
        return None
    try:
        meta = get_meta(result)
        if meta.get("matrix_semantics") == "diag_only_same_layer_matrix":
            raise ValueError(
                "result stores only a diag-only same-layer matrix; use extract_cross_same_layer_scores instead."
            )
        row_states, col_states = get_row_col_states(result)
        matrix = validate_matrix_and_states(result.get("matrix"), row_states, col_states, label=label)
        if not row_states:
            row_states = [f"row_{idx}" for idx in range(matrix.shape[0])]
            warn(f"{label}: row state names were missing; generated placeholder names.")
        if not col_states:
            col_states = [f"col_{idx}" for idx in range(matrix.shape[1])]
            warn(f"{label}: column state names were missing; generated placeholder names.")
        return {
            "matrix": matrix,
            "row_states": row_states,
            "col_states": col_states,
            "meta": meta,
        }
    except Exception as exc:
        warn(f"{label}: could not parse cross_all result: {exc}")
        return None


def extract_cross_same_layer_scores(result, label: str = "cross_same_layer"):
    """Extract same-layer scores.

    Primary sources are ``same_layer_vector`` and ``same_layer_scores`` from
    compute_cka.py. A diag-only matrix is treated only as a compatibility
    fallback and is not the preferred representation.
    """
    if result is None:
        return None
    try:
        meta = get_meta(result)
        row_states, col_states = get_row_col_states(result)

        if "same_layer_vector" in result:
            scores = to_numpy(result["same_layer_vector"]).reshape(-1).astype(float, copy=False)
            states = row_states or col_states
            if not states and isinstance(result.get("same_layer_scores"), Mapping):
                states = list(result["same_layer_scores"].keys())
            if not states:
                states = [f"state_{idx}" for idx in range(scores.shape[0])]
                warn(f"{label}: state names were missing; generated placeholder names.")
            if len(states) != scores.shape[0]:
                raise ValueError(
                    f"same_layer_vector length {scores.shape[0]} != number of states {len(states)}."
                )
            return {
                "states": list(states),
                "scores": scores,
                "meta": meta,
                "source": "same_layer_vector",
            }

        if isinstance(result.get("same_layer_scores"), Mapping):
            score_map = result["same_layer_scores"]
            states = row_states or col_states or list(score_map.keys())
            missing = [state for state in states if state not in score_map]
            if missing:
                warn(f"{label}: same_layer_scores is missing states {missing}; using the intersection.")
            states = [state for state in states if state in score_map]
            if not states:
                raise ValueError("No overlapping states found in same_layer_scores.")
            scores = np.asarray([float(score_map[state]) for state in states], dtype=float)
            return {
                "states": states,
                "scores": scores,
                "meta": meta,
                "source": "same_layer_scores",
            }

        if "matrix" in result:
            warn(
                f"{label}: same_layer_vector/same_layer_scores missing; using matrix diagonal fallback only for compatibility."
            )
            matrix = to_numpy(result["matrix"])
            if matrix.ndim != 2:
                raise ValueError(f"matrix must be 2D, got shape {matrix.shape}.")
            if row_states and col_states:
                common_states = [state for state in row_states if state in set(col_states)]
                if not common_states:
                    raise ValueError("No overlapping state names found between rows and columns.")
                if len(common_states) < len(row_states) or len(common_states) < len(col_states):
                    warn(f"{label}: row/column state names differ; using the intersection for plotting.")
                row_lookup = {state: idx for idx, state in enumerate(row_states)}
                col_lookup = {state: idx for idx, state in enumerate(col_states)}
                scores = np.asarray(
                    [float(matrix[row_lookup[state], col_lookup[state]]) for state in common_states],
                    dtype=float,
                )
                return {
                    "states": common_states,
                    "scores": scores,
                    "meta": meta,
                    "source": "diag_only_matrix_fallback",
                }
            if matrix.shape[0] != matrix.shape[1]:
                raise ValueError("Matrix fallback requires a square matrix when state names are missing.")
            states = row_states or col_states or [f"state_{idx}" for idx in range(matrix.shape[0])]
            scores = np.diag(matrix).astype(float, copy=False)
            if len(states) != scores.shape[0]:
                raise ValueError(
                    f"Diagonal length {scores.shape[0]} != number of states {len(states)}."
                )
            return {
                "states": list(states),
                "scores": scores,
                "meta": meta,
                "source": "diag_only_matrix_fallback",
            }

        raise ValueError("Result does not contain same_layer_vector, same_layer_scores, or matrix.")
    except Exception as exc:
        warn(f"{label}: could not parse cross_same_layer result: {exc}")
        return None


def plot_cka_heatmap(
    matrix,
    row_states,
    col_states,
    title: str,
    *,
    ax=None,
    cmap: str = HEATMAP_CMAP,
    vmin: float = HEATMAP_VMIN,
    vmax: float = HEATMAP_VMAX,
    show_values: bool = SHOW_VALUES_IN_HEATMAP,
    colorbar: bool = True,
    xlabel: str = "Residual state",
    ylabel: str = "Residual state",
):
    """Plot a CKA heatmap using matplotlib only."""
    matrix = np.asarray(matrix, dtype=float)
    if ax is None:
        width = max(5.4, 0.8 * len(col_states) + 1.8)
        height = max(4.8, 0.8 * len(row_states) + 1.8)
        fig, ax = plt.subplots(figsize=(width, height), dpi=FIG_DPI)
    else:
        fig = ax.figure

    aspect = "equal" if matrix.shape[0] == matrix.shape[1] else "auto"
    im = ax.imshow(matrix, cmap=cmap, vmin=vmin, vmax=vmax, aspect=aspect, interpolation="nearest")
    ax.set_title(title, pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(np.arange(len(col_states)))
    ax.set_xticklabels([pretty_state_name(state) for state in col_states], rotation=35, ha="right")
    ax.set_yticks(np.arange(len(row_states)))
    ax.set_yticklabels([pretty_state_name(state) for state in row_states])

    if show_values and matrix.size <= 144:
        for row_idx in range(matrix.shape[0]):
            for col_idx in range(matrix.shape[1]):
                value = matrix[row_idx, col_idx]
                if np.isfinite(value):
                    ax.text(col_idx, row_idx, f"{value:.2f}", ha="center", va="center", fontsize=7.5)

    if colorbar:
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="CKA")

    return fig, ax, im


def plot_same_layer_line(states, scores, title: str, *, ax=None, color: str = "tab:blue"):
    """Plot a same-layer CKA curve with point markers."""
    scores = np.asarray(scores, dtype=float)
    if ax is None:
        width = max(7.0, 0.8 * len(states) + 2.0)
        fig, ax = plt.subplots(figsize=(width, 4.0), dpi=FIG_DPI)
    else:
        fig = ax.figure

    x = np.arange(len(states))
    ax.plot(x, scores, marker="o", linewidth=2.0, markersize=5.5, color=color)
    ax.set_title(title, pad=10)
    ax.set_xlabel("Residual state")
    ax.set_ylabel("CKA")
    ax.set_ylim(0.0, 1.02)
    ax.set_xticks(x)
    ax.set_xticklabels([pretty_state_name(state) for state in states], rotation=35, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.35)
    return fig, ax


def maybe_save_figure(fig, filename: str):
    """Save a figure when SAVE_FIGURES is enabled."""
    if not SAVE_FIGURES:
        return None
    ensure_dir(SAVE_DIR)
    path = SAVE_DIR / with_run_tag(filename)
    fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight")
    SAVED_FIGURES.append(path)
    print(f"[saved] {path}")
    return path


def summarize_matrix(matrix, *, exclude_diagonal: bool = False):
    """Compute a lightweight numeric summary for a matrix."""
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2:
        raise ValueError(f"Expected a 2D matrix, got shape {matrix.shape}.")

    mask = np.isfinite(matrix)
    if exclude_diagonal:
        if matrix.shape[0] != matrix.shape[1]:
            raise ValueError("exclude_diagonal=True requires a square matrix.")
        mask &= ~np.eye(matrix.shape[0], dtype=bool)

    values = matrix[mask]
    if values.size == 0:
        return None

    return {
        "count": int(values.size),
        "mean": float(np.mean(values)),
        "median": float(np.median(values)),
        "min": float(np.min(values)),
        "max": float(np.max(values)),
    }


def summarize_same_layer(states, scores):
    """Compute a lightweight numeric summary for same-layer CKA values."""
    scores = np.asarray(scores, dtype=float)
    if scores.ndim != 1:
        raise ValueError(f"Expected a 1D score vector, got shape {scores.shape}.")
    if scores.size == 0:
        return None
    min_idx = int(np.argmin(scores))
    max_idx = int(np.argmax(scores))
    return {
        "count": int(scores.size),
        "mean": float(np.mean(scores)),
        "median": float(np.median(scores)),
        "lowest_state": states[min_idx],
        "lowest_score": float(scores[min_idx]),
        "highest_state": states[max_idx],
        "highest_score": float(scores[max_idx]),
    }


def summarize_cross_all(matrix, row_states, col_states):
    """For each row, report the most similar column state."""
    matrix = np.asarray(matrix, dtype=float)
    if matrix.ndim != 2:
        raise ValueError(f"Expected a 2D matrix, got shape {matrix.shape}.")
    records = []
    for row_idx, row_state in enumerate(row_states):
        row = matrix[row_idx]
        finite_mask = np.isfinite(row)
        if not finite_mask.any():
            records.append({"row_state": row_state, "best_match": None, "best_score": np.nan})
            continue
        masked_row = np.where(finite_mask, row, -np.inf)
        col_idx = int(np.argmax(masked_row))
        records.append(
            {
                "row_state": row_state,
                "best_match": col_states[col_idx],
                "best_score": float(row[col_idx]),
            }
        )
    return records


def display_records(records, title: str | None = None):
    """Display a list of dicts as a DataFrame when pandas is available."""
    if title:
        print(title)
    if not records:
        print("(no data)")
        return
    if pd is not None:
        display(pd.DataFrame(records))
    else:
        for record in records:
            print(record)


def inspect_result(name: str, result, path: Path):
    """Build a compact inspection record for one loaded result."""
    if result is None:
        return {
            "name": name,
            "loaded": False,
            "path": str(path),
            "mode": None,
            "num_samples_used": None,
            "row_states": "-",
            "col_states": "-",
            "matrix_shape": None,
            "same_layer_vector_shape": None,
            "group_filter": None,
            "result_structure": None,
            "matrix_semantics": None,
            "artifact_a": None,
            "artifact_b": None,
            "summary_tag": "-",
        }

    meta = get_meta(result)
    row_states, col_states = get_row_col_states(result)

    return {
        "name": name,
        "loaded": True,
        "path": str(path),
        "mode": infer_mode(result),
        "num_samples_used": meta.get("num_samples_used"),
        "row_states": format_state_list(row_states),
        "col_states": format_state_list(col_states),
        "matrix_shape": shape_or_none(result.get("matrix")),
        "same_layer_vector_shape": shape_or_none(result.get("same_layer_vector")),
        "group_filter": meta.get("group_filter"),
        "result_structure": meta.get("result_structure"),
        "matrix_semantics": meta.get("matrix_semantics"),
        "artifact_a": meta.get("artifact_a"),
        "artifact_b": meta.get("artifact_b"),
        "summary_tag": safe_title_from_meta(result),
    }


## 5. Inspect Loaded Results / Sanity Check

In [ ]:
print("Configured result files:")
for name, path in RESULT_PATHS.items():
    status = "found" if Path(path).is_file() else "missing"
    print(f"- {name}: {path} [{status}]")

RESULTS = {name: load_pt(path) for name, path in RESULT_PATHS.items()}

inspection_rows = [
    inspect_result(name, result, RESULT_PATHS[name])
    for name, result in RESULTS.items()
]

display_records(inspection_rows, title="Loaded compute_cka outputs")

for row in inspection_rows:
    print("-" * 100)
    print(f"name: {row['name']}")
    print(f"loaded: {row['loaded']}")
    print(f"path: {row['path']}")
    print(f"mode: {row['mode']}")
    print(f"num_samples_used: {row['num_samples_used']}")
    print(f"row_states: {row['row_states']}")
    print(f"col_states: {row['col_states']}")
    print(f"matrix_shape: {row['matrix_shape']}")
    print(f"same_layer_vector_shape: {row['same_layer_vector_shape']}")
    print(f"group_filter: {row['group_filter']}")
    print(f"result_structure: {row['result_structure']}")
    print(f"matrix_semantics: {row['matrix_semantics']}")
    print(f"artifact_a: {row['artifact_a']}")
    print(f"artifact_b: {row['artifact_b']}")
    print(f"summary_tag: {row['summary_tag']}")


## 6. Plot: Within-model CKA

In [ ]:
within_standard = extract_within_matrix(RESULTS.get("within_standard"), label="within_standard")
within_attnres = extract_within_matrix(RESULTS.get("within_attnres"), label="within_attnres")

if within_standard is not None:
    fig, ax, _ = plot_cka_heatmap(
        within_standard["matrix"],
        within_standard["row_states"],
        within_standard["col_states"],
        title="Within-model CKA: standard",
        xlabel="Residual state",
        ylabel="Residual state",
    )
    maybe_save_figure(fig, "within_standard.png")
    plt.show()
else:
    print("Skipped standard within-model heatmap: result missing or invalid.")

if within_attnres is not None:
    fig, ax, _ = plot_cka_heatmap(
        within_attnres["matrix"],
        within_attnres["row_states"],
        within_attnres["col_states"],
        title="Within-model CKA: attnres",
        xlabel="Residual state",
        ylabel="Residual state",
    )
    maybe_save_figure(fig, "within_attnres.png")
    plt.show()
else:
    print("Skipped attnres within-model heatmap: result missing or invalid.")

if within_standard is not None and within_attnres is not None:
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8), dpi=FIG_DPI, constrained_layout=True)
    fig.suptitle("Within-model CKA comparison", fontsize=14)
    _, _, im = plot_cka_heatmap(
        within_standard["matrix"],
        within_standard["row_states"],
        within_standard["col_states"],
        title="Within-model CKA: standard",
        ax=axes[0],
        colorbar=False,
        xlabel="Residual state",
        ylabel="Residual state",
    )
    plot_cka_heatmap(
        within_attnres["matrix"],
        within_attnres["row_states"],
        within_attnres["col_states"],
        title="Within-model CKA: attnres",
        ax=axes[1],
        colorbar=False,
        xlabel="Residual state",
        ylabel="Residual state",
    )
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.85, label="CKA")
    maybe_save_figure(fig, "within_side_by_side.png")
    plt.show()
else:
    print("Skipped within-model side-by-side comparison: need both standard and attnres results.")


## 7. Plot: Cross-model same-layer CKA

In [ ]:
cross_same_layer = extract_cross_same_layer_scores(
    RESULTS.get("cross_same_layer"),
    label="cross_same_layer",
)

if cross_same_layer is not None:
    fig, ax = plot_same_layer_line(
        cross_same_layer["states"],
        cross_same_layer["scores"],
        title="Same-layer CKA: standard vs attnres",
        color="tab:blue",
    )
    maybe_save_figure(fig, "same_layer_standard_vs_attnres.png")
    plt.show()
else:
    print("Skipped standard vs attnres same-layer plot: result missing or invalid.")


## 8. Plot: Across-checkpoint same-layer CKA

In [ ]:
checkpoint_standard = extract_cross_same_layer_scores(
    RESULTS.get("checkpoint_standard"),
    label="checkpoint_standard",
)
checkpoint_attnres = extract_cross_same_layer_scores(
    RESULTS.get("checkpoint_attnres"),
    label="checkpoint_attnres",
)

if checkpoint_standard is not None:
    fig, ax = plot_same_layer_line(
        checkpoint_standard["states"],
        checkpoint_standard["scores"],
        title=CHECKPOINT_STANDARD_TITLE,
        color="tab:green",
    )
    maybe_save_figure(fig, "checkpoint_standard_same_layer.png")
    plt.show()
else:
    print("Skipped standard across-checkpoint plot: result missing or invalid.")

if checkpoint_attnres is not None:
    fig, ax = plot_same_layer_line(
        checkpoint_attnres["states"],
        checkpoint_attnres["scores"],
        title=CHECKPOINT_ATTNRES_TITLE,
        color="tab:orange",
    )
    maybe_save_figure(fig, "checkpoint_attnres_same_layer.png")
    plt.show()
else:
    print("Skipped attnres across-checkpoint plot: result missing or invalid.")

if checkpoint_standard is not None and checkpoint_attnres is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.2), dpi=FIG_DPI, constrained_layout=True, sharey=True)
    fig.suptitle("Same-layer CKA across checkpoints", fontsize=14)
    plot_same_layer_line(
        checkpoint_standard["states"],
        checkpoint_standard["scores"],
        title="standard",
        ax=axes[0],
        color="tab:green",
    )
    plot_same_layer_line(
        checkpoint_attnres["states"],
        checkpoint_attnres["scores"],
        title="attnres",
        ax=axes[1],
        color="tab:orange",
    )
    maybe_save_figure(fig, "checkpoint_side_by_side.png")
    plt.show()
else:
    print("Skipped checkpoint side-by-side comparison: need both standard and attnres results.")


## 9. Plot: Cross-all matrix (optional)

In [ ]:
cross_all_standard_attnres = extract_cross_all_matrix(
    RESULTS.get("cross_all_standard_attnres"),
    label="cross_all_standard_attnres",
)

if cross_all_standard_attnres is not None:
    fig, ax, _ = plot_cka_heatmap(
        cross_all_standard_attnres["matrix"],
        cross_all_standard_attnres["row_states"],
        cross_all_standard_attnres["col_states"],
        title="Cross-model all-pairs CKA: standard vs attnres",
        xlabel="AttnRes residual state",
        ylabel="Standard residual state",
    )
    maybe_save_figure(fig, "cross_all_standard_vs_attnres.png")
    plt.show()
else:
    print("Skipped cross-all heatmap: result missing or invalid.")


## 10. Lightweight Numeric Summary

In [ ]:
within_records = []
for name, payload in [("standard", within_standard), ("attnres", within_attnres)]:
    if payload is None:
        continue
    summary = summarize_matrix(payload["matrix"], exclude_diagonal=True)
    if summary is None:
        continue
    within_records.append(
        {
            "model": name,
            "mean_offdiag": summary["mean"],
            "median_offdiag": summary["median"],
            "min_offdiag": summary["min"],
            "max_offdiag": summary["max"],
            "num_pairs": summary["count"],
        }
    )

same_layer_label_map = {
    "standard_vs_attnres": "standard vs attnres",
    "checkpoint_standard": "checkpoint comparison: standard",
    "checkpoint_attnres": "checkpoint comparison: attnres",
}

same_layer_records = []
for name, payload in [
    ("standard_vs_attnres", cross_same_layer),
    ("checkpoint_standard", checkpoint_standard),
    ("checkpoint_attnres", checkpoint_attnres),
]:
    if payload is None:
        continue
    summary = summarize_same_layer(payload["states"], payload["scores"])
    if summary is None:
        continue
    same_layer_records.append(
        {
            "comparison": same_layer_label_map[name],
            "source": payload.get("source"),
            "mean": summary["mean"],
            "median": summary["median"],
            "lowest_state": pretty_state_name(summary["lowest_state"]),
            "lowest_score": summary["lowest_score"],
            "highest_state": pretty_state_name(summary["highest_state"]),
            "highest_score": summary["highest_score"],
            "num_states": summary["count"],
        }
    )

display_records(within_records, title="Within-model off-diagonal CKA summary")
display_records(same_layer_records, title="Same-layer CKA summary")

print("\nText summary:")
for record in within_records:
    print(
        f"- {record['model']}: mean off-diagonal={record['mean_offdiag']:.3f}, "
        f"median={record['median_offdiag']:.3f}, range=[{record['min_offdiag']:.3f}, {record['max_offdiag']:.3f}]"
    )

for record in same_layer_records:
    print(
        f"- {record['comparison']}: mean={record['mean']:.3f}, "
        f"lowest={record['lowest_state']} ({record['lowest_score']:.3f}), "
        f"highest={record['highest_state']} ({record['highest_score']:.3f})"
    )

if cross_all_standard_attnres is not None:
    cross_all_records = summarize_cross_all(
        cross_all_standard_attnres["matrix"],
        cross_all_standard_attnres["row_states"],
        cross_all_standard_attnres["col_states"],
    )
    cross_all_records = [
        {
            "row_state": pretty_state_name(record["row_state"]),
            "best_match": None if record["best_match"] is None else pretty_state_name(record["best_match"]),
            "best_score": record["best_score"],
        }
        for record in cross_all_records
    ]
    display_records(cross_all_records, title="Cross-all best-match summary (row -> best column)")
    for record in cross_all_records:
        print(f"- {record['row_state']} -> {record['best_match']} ({record['best_score']:.3f})")
else:
    print("Cross-all summary skipped: result missing or invalid.")


## 11. Interpretation Notes

How to read these figures:

- CKA measures representational similarity rather than performance.
- Higher CKA means two representations organize aligned samples more similarly.
- Lower CKA means the compared representations are more divergent in geometry.
- Within-model off-diagonal structure reflects how similar or differentiated residual states remain across depth.
- Cross-model same-layer CKA reflects how similarly `standard` and `attnres` organize information at the same nominal depth.
- Across-checkpoint same-layer CKA reflects how stable a model's layerwise representational geometry remains over training.
- Cross-all heatmaps help identify whether one layer aligns most strongly with the matched depth or with a different depth in the other artifact.

Interpretation should stay cautious:

- Lower same-layer CKA suggests stronger representational divergence between architectures at the same depth, not better performance.
- Lower within-model off-diagonal CKA can be consistent with stronger layerwise differentiation, but it does not by itself establish a superior representation strategy.
- If `attnres` changes how block inputs are organized, that may appear as lower cross-model same-layer CKA or a different within-model layerwise pattern.
- These patterns may be consistent with different information-organization strategies, but they do not by themselves establish causality, purity, or stronger memorization.


## 12. Export Figures

In [ ]:
print(f"RUN_TAG = {RUN_TAG!r}")
print(f"SAVE_FIGURES = {SAVE_FIGURES}")
print(f"SAVE_DIR = {SAVE_DIR}")
print(f"RESULTS_DIR = {RESULTS_DIR}")

if SAVE_FIGURES:
    ensure_dir(SAVE_DIR)
    if SAVED_FIGURES:
        print("Saved figure files:")
        for path in SAVED_FIGURES:
            print(f"- {path}")
    else:
        print("No figures were saved. Either the plotting cells were not run yet or the corresponding inputs were missing.")
else:
    print("Figure export is disabled. Set SAVE_FIGURES = True in the config cell to save PNG files.")
